In [ ]:
# CELL 0
# MOUNTING DRIVE

from google.colab import drive
drive.mount('/content/drive')

# ----------------- CHECKING PATH TO AN IMAGE

import os

# insert the name the your Drive folder which contains a shortcut to the project data
my_drive_folder = "APAI_CVDL_shared"

# expected True if path exists
print(os.path.exists(f"/content/drive/MyDrive/{my_drive_folder}/project/data/pre/img/10674_train_0_.npy"))

In [ ]:
# CELL 1
# UNZIPPING DATA TO /content/pre

import zipfile
from pathlib import Path
from tqdm import tqdm

DRIVE_ZIP = Path(f"/content/drive/MyDrive/{my_drive_folder}/project/data/pre.zip")
EXTRACT_ROOT = Path("/content/data")

if not EXTRACT_ROOT.exists(): # heuristic, but good enough
    # create new local data dir if not already present
    EXTRACT_ROOT.mkdir(parents=True, exist_ok=True)

    # open the zip from Drive in read mode
    with zipfile.ZipFile(DRIVE_ZIP, "r") as z:
        # list of zipped objects
        objects = z.infolist()

        # total number of zipped objects
        total = len(objects)

        for o in tqdm(objects, total=total, desc="unzipping"):
            # unzip
            z.extract(o, EXTRACT_ROOT)
else:
    print("data already extracted, skipping step")

In [ ]:
# CELL 2
# LOADING THE MANIFEST

import os

# build the path to the correct manifest file
META_DIR = f"/content/drive/MyDrive/{my_drive_folder}/project/meta"

# build the path to the correct manifest file
manifest_path = f"{META_DIR}/manifest.parquet"

# expected True if path exists
print(os.path.exists(manifest_path))

In [ ]:
# CELL 3
# BUILDING THE DATAFRAME

import pandas as pd

# build DataFrame
df = pd.read_parquet(manifest_path)

# print row count and col names from the manifest
print("rows in manifest_preproc.parquet:", len(df))
print("cols in manifest_preproc.parquet:", df.columns.tolist())

# print first 5 rows
df.head(5)

In [ ]:
# CELL 4
# VERIFYING DATA TYPES

import os
import numpy as np
import matplotlib.pyplot as plt

# define root
ROOT = "/content"

# find first row with a positive mask
idx = 0
for i, row in df.iterrows():
    if "train_1" in row["mask_path"]:
        print("row index:", i)
        idx = i
        break

# assign index
row = df.iloc[idx]

# ----------------- ADAPTING SLASHES TO LINUX

# replace slashes
img_rel_path = row["img_path"].replace("\\", "/")
mask_rel_path = row["mask_path"].replace("\\", "/")

# join paths
img_path  = os.path.join(ROOT, str(img_rel_path))
mask_path = os.path.join(ROOT, str(mask_rel_path))

# check
print("img_path:", img_path)
print("mask_path:", mask_path)

# ----------------- PRINTING AND PLOTTING TO VERIFY DATA

# images and masks from disk to memory
img = np.load(img_path)
mask = np.load(mask_path)

# print dtype info
print("\nIMG  shape/dtype/min/max:", img.shape, img.dtype, img.min(), img.max())
print("MASK shape/dtype/min/max:", mask.shape, mask.dtype, mask.min(), mask.max())
print("MASK unique (first 20):", np.unique(mask)[:20])

# plot
plt.figure()
plt.imshow(img, cmap="gray")
plt.title("Image")
plt.colorbar()
plt.show()

plt.figure()
plt.imshow(mask, cmap="gray")
plt.title("Mask")
plt.colorbar()
plt.show()

In [ ]:
# CELL 5
# CREATING SPLIT DATAFRAMES

# train split
df_train = df[df["split"] == "train"].copy()
# print rows
print("train rows:", len(df_train))

# val split
df_val = df[df["split"] == "val"].copy()
# print rows
print("val rows:", len(df_val))

In [7]:
# CELL 6
# CREATING THE DATASET CLASS

import os
import numpy as np
import torch
from torch.utils.data import Dataset

class BaselineDataset(Dataset):
    def __init__(self, df, root, img_col="img_path", mask_col="mask_path"):
        self.df = df.reset_index(drop=True)
        self.root = root
        self.img_col = img_col
        self.mask_col = mask_col

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        # replace slashes and join paths
        img_rel_path = row[self.img_col].replace("\\", "/")
        mask_rel_path = row[self.mask_col].replace("\\", "/")
        img_path  = os.path.join(self.root, str(img_rel_path))
        mask_path = os.path.join(self.root, str(mask_rel_path))

        # load image and mask given path AND convert them to float32
        img = np.load(img_path).astype(np.float32)
        mask = np.load(mask_path).astype(np.float32)

        # from (H, W) to (1, H, W)
        img = torch.from_numpy(img).unsqueeze(0)
        mask = torch.from_numpy(mask).unsqueeze(0)

        return img, mask

In [8]:
# CELL 7
# BUILDING DATASETS

# training dataset
train_ds = BaselineDataset(df_train, root=ROOT)

# validation dataset
val_ds = BaselineDataset(df_val, root=ROOT)

In [ ]:
# CELL 8
# BUILDING DATALOADERS, IMPLEMENTING OVERSAMPLING

from torch.utils.data import DataLoader, WeightedRandomSampler
import numpy as np
import torch

# oversampling flag
oversample = True

# oversampling multiplier
k = 0.5

# generate a numpy array of ints corresponding to classes of samples in the train split
# note: label[i] corresponds to dataset item i, because Dataset resets indexes
labels = df_train["class"].to_numpy().astype(np.int64)

# get the number of positive and negative samples
n_pos = labels.sum()
assert n_pos > 0
n_neg = len(labels) - n_pos

if oversample == True:
    # set weights
    w_pos = (n_neg/n_pos) * k # note: a weight of N means that sample has N times the chance of being picked by the loader
    w_neg = 1.0

    # create array of weights per index of labels array
    weights = np.where(labels == 1, w_pos, w_neg).astype(np.float64) # int Numpy array

    # initialize sampler
    sampler = WeightedRandomSampler(
        weights=torch.from_numpy(weights),
        num_samples=len(weights),
        replacement=True
    )

    # build oversampling dataloader
    train_loader = DataLoader(
        train_ds,
        batch_size=8,
        sampler=sampler,
        shuffle=False,
        num_workers=2,
        pin_memory=True,
        drop_last=True
    )

    # print oversampling status and checks
    print(
        "OVERSAMPLING ACTIVE | ",
        f"k ={k} | ",
        f"w_pos ={w_pos}"
    )
else:
    # build standard, non-oversampling dataloader
    train_loader = DataLoader(
        train_ds,
        batch_size=8,
        shuffle=True,
        num_workers=2,
        pin_memory=True,
        drop_last=True
    )
    # print oversampling status
    print("OVERSAMPLING INACTIVE")

# validation dataloader
val_loader = DataLoader(
    val_ds,
    batch_size=8,
    shuffle=False, # deterministic validation
    num_workers=2,
    pin_memory=True,
    drop_last=False
)

# ----------------- CHECK (expexted out: (B, 1, 512, 512) float32)

import torch

imgs, masks = next(iter(train_loader))
print("imgs:", imgs.shape, imgs.dtype, "min/max:", imgs.min().item(), imgs.max().item())
print("masks:", masks.shape, masks.dtype, "unique:", torch.unique(masks))

In [10]:
# CELL 9
# MODEL DEFINITION

import torch.nn as nn
from torch.nn.init import kaiming_normal_, zeros_

# define function for two consecutive 3x3 convs
def double_convolution(in_channels, out_channels, num_groups=8):
    # for GroupNorm, ensure num_groups divides out_channels
    g = min(num_groups, out_channels)
    while out_channels % g != 0:
        g -= 1

    conv_op = nn.Sequential(
        nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
        nn.GroupNorm(g, out_channels), # GroupNorm
        nn.ReLU(inplace=True),

        nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
        nn.GroupNorm(g, out_channels), # GroupNorm
        nn.ReLU(inplace=True)
    )
    return conv_op

class UNet(nn.Module):
    def __init__(self, num_classes):
        super(UNet, self).__init__()

        # define pooling layer
        self.max_pool2d = nn.MaxPool2d(kernel_size=2, stride=2)

        # define double convs for encoder
        self.down_convolution_1 = double_convolution(1, 64) # in encoder stage 1
        self.down_convolution_2 = double_convolution(64, 128) # in encoder stage 2
        self.down_convolution_3 = double_convolution(128, 256) # in encoder stage 3
        self.down_convolution_4 = double_convolution(256, 512) # in encoder stage 4
        self.down_convolution_5 = double_convolution(512, 1024) # in bottleneck

        # ----------------- DEFINE DECODER
        # in decoder stage 1
        self.up_transpose_1 = nn.ConvTranspose2d(
            in_channels=1024,
            out_channels=512,
            kernel_size=2,
            stride=2
        )
        self.up_convolution_1 = double_convolution(1024, 512)

        # in decoder stage 2
        self.up_transpose_2 = nn.ConvTranspose2d(
            in_channels=512,
            out_channels=256,
            kernel_size=2,
            stride=2
        )
        self.up_convolution_2 = double_convolution(512, 256)

        # in decoder stage 3
        self.up_transpose_3 = nn.ConvTranspose2d(
            in_channels=256,
            out_channels=128,
            kernel_size=2,
            stride=2
        )
        self.up_convolution_3 = double_convolution(256, 128)

        # in decoder stage 4
        self.up_transpose_4 = nn.ConvTranspose2d(
            in_channels=128,
            out_channels=64,
            kernel_size=2,
            stride=2
        )
        self.up_convolution_4 = double_convolution(128, 64)

        # ----------------- DEFINE OUTPUT
        self.out = nn.Conv2d(
            in_channels=64,
            out_channels=num_classes,
            kernel_size=1
        )

    def forward(self, x):
        # ----------------- ENCODER
        down_1 = self.down_convolution_1(x)
        down_2 = self.max_pool2d(down_1)

        down_3 = self.down_convolution_2(down_2)
        down_4 = self.max_pool2d(down_3)

        down_5 = self.down_convolution_3(down_4)
        down_6 = self.max_pool2d(down_5)

        down_7 = self.down_convolution_4(down_6)
        down_8 = self.max_pool2d(down_7)

        down_9 = self.down_convolution_5(down_8)

        # ----------------- DECODER
        up_1 = self.up_transpose_1(down_9)
        x = self.up_convolution_1(torch.cat([down_7, up_1], 1))

        up_2 = self.up_transpose_2(x)
        x = self.up_convolution_2(torch.cat([down_5, up_2], 1))

        up_3 = self.up_transpose_3(x)
        x = self.up_convolution_3(torch.cat([down_3, up_3], 1))

        up_4 = self.up_transpose_4(x)
        x = self.up_convolution_4(torch.cat([down_1, up_4], 1))

        # ----------------- OUTPUT
        out = self.out(x)
        return out

In [ ]:
# CELL 10
# MODEL CHECK
# just checking shapes and dtypes of batches and logits

import torch

# select GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

# instantiate model, move to device
model = UNet(num_classes=1).to(device)

# get one batch from dataloader
imgs, masks = next(iter(train_loader))

# move images to device
imgs = imgs.to(device)

# forward pass
with torch.no_grad():
    logits = model(imgs)

# print to check shape correctness (expected: (B, 1, 512, 512))
print(
    f"imgs shape:{imgs.shape} | ",
    f"masks shape:{masks.shape} | ",
    f"logits shape:{logits.shape}"
)
# print to check dtype correctness (expected: float32)
print(
    f"imgs dtype:{imgs.dtype} | ",
    f"masks dtype:{masks.dtype} | ",
    f"logits dtype:{logits.dtype}"
)

In [12]:
# CELL 11
# DEFINING LOSSES

import torch
import torch.nn as nn
import torch.nn.functional as F

def dice_loss(logits, gt, eps=1e-6):
    # convert logits to probabilities
    probs = torch.sigmoid(logits) # (B, 1, H, W)

    # flattening
    probs = probs.flatten(start_dim=1) # (B, H*W)
    gt = gt.flatten(start_dim=1) # (B, H*W)

    # compute intersection per sample, for DSC
    intersection = (probs * gt).sum(dim=1) # (B,), TP

    # compute sums per sample, for DSC
    probs_sum = probs.sum(dim=1) # (B,)
    gt_sum = gt.sum(dim=1) # (B,)

    # DSC per sample
    dsc = (2.0 * intersection + eps) / (probs_sum + gt_sum + eps) # (B,)

    # Dice loss per sample
    return 1.0 - dsc # (B,)

class FocalBCEDiceLoss(nn.Module):
    def __init__(self, dice_weight=1.0, alpha=0.25, gamma=2.0, neg_ohem_weight=0.05, neg_topk=1024):
        super().__init__()
        self.dice_weight = dice_weight
        self.alpha = alpha
        self.gamma = gamma
        self.neg_ohem_weight = neg_ohem_weight
        self.neg_topk = neg_topk

    def forward(self, logits, gt):
        gt = gt.float()

        # ----------------- FOCAL BCE WITH LOGITS, ON POSITIVES AND NEGATIVES

        bce = F.binary_cross_entropy_with_logits(logits, gt, reduction="none") # (B, 1, H, W)

        # convert logits to probabilities
        probs = torch.sigmoid(logits) # (B, 1, H, W)

        # pt = probability of the true class
        pt = probs * gt + (1.0 - probs) * (1.0 - gt) # (B, 1, H, W)

        # defin alpha weighting
        alpha_t = self.alpha * gt + (1.0 - self.alpha) * (1.0 - gt) # (B, 1, H, W)

        # focal BCE formula
        focal_bce = alpha_t * (1.0 - pt).pow(self.gamma) * bce # (B, 1, H, W)

        # convert to scalar
        focal_bce = focal_bce.mean() # scalar
        assert focal_bce.ndim == 0, "focal_bce should be a scalar"

        # ----------------- DICE LOSS, ON POSITIVES ONLY

        # Boolean vector the same shape as a batch to restrict Dice computation to positives only
        is_pos = gt.sum(dim=(1, 2, 3)) > 0 # (B,) bool

        if is_pos.any():
            # compute Dice loss on pos samples only, create tensor with the same length of # pos in the batch
            dice_per_sample = dice_loss(logits[is_pos], gt[is_pos]) # (Bpos,)

            # define scalar that averages dice_per_sample losses
            dice = dice_per_sample.mean() # scalar
        else:
            # if there is no positive in the batch, loss is 0.0
            dice = logits.new_tensor(0.0)

        # ----------------- OHEM TOP-K BCE, ON NEGATIVES ONLY

        # to restrict Dice computation to negatives only
        is_neg = ~is_pos # (B,) bool

        if is_neg.any() and self.neg_ohem_weight > 0:
            # select only BCE loss values for neg samples in the batch
            bce_neg = bce[is_neg] # (Bneg, 1, H, W)

            # flatten pixels, to later select the ones with the highest BCE loss
            flat = bce_neg.flatten(start_dim=1) # (Bneg, H*W)

            # define k, make sure it's never above H*W
            k = min(self.neg_topk, flat.shape[1])

            # for each negative, pick the k pixels with the highest BCE loss
            topk_vals, _ = torch.topk(flat, k, dim=1) # (Bneg, k)

            # average per batch
            neg_ohem = topk_vals.mean()
        else:
            # if there is no negative in the batch, loss is 0.0
            neg_ohem = logits.new_tensor(0.0)

        # total loss
        return focal_bce + self.dice_weight * dice + self.neg_ohem_weight * neg_ohem

# initialize criterion instance
criterion = FocalBCEDiceLoss(
    dice_weight=1.0,
    alpha=0.25,
    gamma=2.0,
    neg_ohem_weight=0.05,
    neg_topk=1024
).to(device)

In [13]:
# CELL 12
# TRAINING LOOP

import torch
from tqdm import tqdm

# define optimizer
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-4
)

# define scheduler
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.5,
    patience=1,
    threshold=1e-3,
    min_lr=1e-6
)

# for each epoch...
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    count_batches = 0

    # for each batch...
    for imgs, masks in tqdm(loader, desc="train", leave=False):
        # moving tensors to device
        imgs, masks = imgs.to(device), masks.to(device).float()

        # clear gradients
        optimizer.zero_grad()

        # forward pass
        logits = model(imgs)

        # compute loss
        loss = criterion(logits, masks)

        # backpropagation
        loss.backward()

        # update weights
        optimizer.step()

        running_loss += loss.item()
        count_batches += 1

    # return avg train loss in an epoch for logging
    return running_loss / max(count_batches, 1)

In [14]:
# CELL 13
# VALIDATION LOOP

import torch
from tqdm import tqdm

# for each epoch...
@torch.no_grad()
def validate_one_epoch(model, loader, criterion, device, threshold=0.5, eps=1e-6):
    model.eval()

    # loss bookkeeping
    running_loss = 0.0
    count_batches = 0

    # metrics bookkeeping
    dsc_sum = 0.0 # sum of DSC over positive samples
    iou_sum = 0.0 # sum of IoU over positive samples
    pos_count = 0 # number of positive samples evaluated

    neg_total = 0 # number of negative samples evaluated
    neg_fp = 0 # how many of those had false positives (thresholded)
    neg_fphw_sum = 0.0 # sum over negative samples of FP pixels/HW (thresholded)

    # for each batch...
    for imgs, masks in tqdm(loader, desc="val", leave=False):
        # moving tensors to device
        imgs, masks = imgs.to(device), masks.to(device).float()

        # forward pass
        logits = model(imgs)

        # compute loss
        loss = criterion(logits, masks)

        running_loss += loss.item()
        count_batches += 1

        # convert logits to binary predictions
        probs = torch.sigmoid(logits) # (B, 1, H, W)

        # apply threshold
        preds = (probs > threshold).float() # (B, 1, H, W)

        # identify pos/neg samples by gt
        is_pos = (masks.sum(dim=(1, 2, 3)) > 0) # bool tensor (B,)
        is_neg = ~is_pos # bool tensor (B,)

        # only compute DSC and IoU if there is at least one positive image in the batch
        if is_pos.any():
            # selecting positive samples
            p = preds[is_pos] # (B_pos, 1, H, W)
            g = masks[is_pos] # (B_pos, 1, H, W)
            # flattening
            p = p.flatten(start_dim=1) # (B_pos, H*W)
            g = g.flatten(start_dim=1) # (B_pos, H*W)

            # compute intersection per sample
            intersection = (p * g).sum(dim=1) # (B_pos,), TP pixels
            # compute sums per sample
            p_sum = p.sum(dim=1) # (B_pos,), predicted foreground pixels
            g_sum = g.sum(dim=1) # (B_pos,), GT foreground pixels
            # compute union per sample
            union = (p + g - p * g).sum(dim=1) # (B_pos), TP + FP + FN

            # compute DSC
            dsc = (2.0 * intersection + eps) / (p_sum + g_sum + eps) # (B_pos,)
            # compute IoU
            iou = (intersection + eps) / (union + eps) # (B_pos)

            # aggregate
            dsc_sum += dsc.sum().item()
            iou_sum += iou.sum().item()
            pos_count += dsc.numel()

        # compute FPIR and FP/HW
        if is_neg.any():
            preds_neg = preds[is_neg]

            # ----------------- FPIR ON NEGATIVES ONLY

            # add number of negatives samples in the batch to bookkeping
            neg_total += is_neg.sum().item()
            # add number of false positives to bookkeeping
            neg_fp += (preds_neg.sum(dim=(1,2,3)) > 0).sum().item()

            # ----------------- FP/HW ON NEGATIVS ONLY

            # flatten
            neg_frac = preds_neg.flatten(start_dim=1) # (B_neg, H*W)
            # mean over H*W, total pixels
            neg_frac = neg_frac.mean(dim=1) # (B_neg,)
            # add to bookkeeping
            neg_fphw_sum += neg_frac.sum().item()

    # define metrics per epoch
    val_loss = running_loss / max(count_batches, 1)
    val_dsc = dsc_sum / max(pos_count, 1)
    val_iou = iou_sum / max(pos_count, 1)

    val_fpir = neg_fp / max(neg_total, 1)
    val_fphw = neg_fphw_sum / max(neg_total, 1)

    # return a dictionary with validation loss and every metric
    return {
        "val_loss": val_loss,
        "val_dsc": val_dsc,
        "val_iou": val_iou,
        "val_fpir": val_fpir,
        "val_fphw": val_fphw
    }

In [ ]:
# CELL 14
# TRAINING

from pathlib import Path
from datetime import datetime
import torch

# creating checkpoint dir
timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
CKPT_DIR = Path(f"/content/drive/MyDrive/{my_drive_folder}/checkpoints/{timestamp}_run")
CKPT_DIR.mkdir(parents=True, exist_ok=True)
print("Checkpoint directory created at", CKPT_DIR)

# choose epoch count
"""
important note: warmup for OHEM takes 4 epochs, so
it doesn't make sense to test for less than ~7-8 epochs

plus, checkpoints as I implemented them are every 4 epochs
"""
num_epochs = 20

# threshold sweep
thresholds = [0.25, 0.5, 0.7]
assert len(thresholds) > 0

train_losses = []
val_losses = []

for epoch in range(num_epochs):
    # OHEM warmup schedule
    ohem_activation_epoch = 4
    if epoch < ohem_activation_epoch:
        criterion.neg_ohem_weight = 0.0
    else:
        criterion.neg_ohem_weight = 0.05

    # run training
    train_loss = train_one_epoch(model, train_loader, criterion, optimizer, device)

    # append training loss to list
    train_losses.append(train_loss)

    for t in thresholds:
        # run validation
        val_results = validate_one_epoch(model, val_loader, criterion, device, threshold=t)

        if t == thresholds[0]:
            # get the val loss for that epoch
            # note: any threshold is fine, but only one must be chosen; I chose the first
            val_loss = val_results["val_loss"]

            # append loss
            val_losses.append(val_loss)

            # step LR scheduler
            scheduler.step(val_loss)

            # print epoch # and losses (printed once every epoch)
            print(
                "\n-------\n"
                f"EPOCH {epoch+1}/{num_epochs} | "
                f"TRAIN LOSS={train_loss:.4f} | "
                f"VAL LOSS={val_results['val_loss']:.4f} | "
                f"LR={optimizer.param_groups[0]['lr']} | "
                f"OHEM WEIGHT={criterion.neg_ohem_weight}"
            )

        # print metrics (printed once per threshold value every epoch)
        print(
            f"THRESHOLD={t}\n"
            f"DSC={val_results['val_dsc']:.4f} | "
            f"IoU={val_results['val_iou']:.4f} | "
            f"FPIR={val_results['val_fpir']:.4f} | "
            f"FPHW={val_results['val_fphw']:.6f}"
        )

    # save basic checkpoints every 5 epochs
    if (epoch + 1) % 4 == 0:
        torch.save({
            "epoch": epoch + 1,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "val_metrics": val_results,
        },
        CKPT_DIR / f"epoch_{epoch+1:03d}.pt")

# ----------------- PLOTTING EPOCH LOSS CURVES

import matplotlib.pyplot as plt

plt.figure()
plt.plot(train_losses, label="train")
plt.plot(val_losses, label="val")
plt.xlabel("epoch")
plt.ylabel("loss")
plt.legend()
plt.show()